In [1]:
%pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install astroquery pyvo
!pip install pyvo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 1.7 MB/s  0:00:06m 1.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.7 MB/s  0:00:00m 1.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 1.6 MB/s  0:00:03m 1.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 1.5 MB/s  0:00:01m 1.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 1.7 MB/s  0:00:031.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [astroquery]7m━━━ 11/12 [astroquery]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
zsh:1: command not found: pip


In [4]:
from astroquery.jplhorizons import Horizons
import pandas as pd
import numpy as np
import pyvo as vo

# Test generate coords for Eureka

In [ ]:

start = '2018-01-01' # yyyy-mm-dd format 
stop = '2023-01-01'
step = '1d' # Step size

obj = Horizons(id='Eureka', location='Q55',
               epochs={'start':start, 'stop':stop,
                       'step':step})

In [125]:
eph = obj.ephemerides()

In [ ]:
# See type returned
print(eph.dtype)

[('targetname', '<U21'), ('datetime_str', '<U17'), ('datetime_jd', '<f8'), ('H', '<f8'), ('G', '<f8'), ('solar_presence', '<U1'), ('lunar_presence', '<U1'), ('RA', '<f8'), ('DEC', '<f8'), ('RA_app', '<f8'), ('DEC_app', '<f8'), ('RA_rate', '<f8'), ('DEC_rate', '<f8'), ('AZ', '<f8'), ('EL', '<f8'), ('AZ_rate', '<f8'), ('EL_rate', '<f8'), ('sat_X', '<f8'), ('sat_Y', '<f8'), ('sat_PANG', '<f8'), ('siderealtime', '<f8'), ('airmass', '<f8'), ('magextinct', '<f8'), ('V', '<f8'), ('surfbright', '<f8'), ('illumination', '<f8'), ('illum_defect', '<f8'), ('sat_sep', '<f8'), ('sat_vis', '<U1'), ('ang_width', '<f8'), ('PDObsLon', '<i8'), ('PDObsLat', '<i8'), ('PDSunLon', '<i8'), ('PDSunLat', '<i8'), ('SubSol_ang', '<f8'), ('SubSol_dist', '<f8'), ('NPole_ang', '<i8'), ('NPole_dist', '<i8'), ('EclLon', '<f8'), ('EclLat', '<f8'), ('r', '<f8'), ('r_rate', '<f8'), ('delta', '<f8'), ('delta_rate', '<f8'), ('lighttime', '<f8'), ('vel_sun', '<f8'), ('vel_obs', '<f8'), ('elong', '<f8'), ('elongFlag', '<U2')

# Connect to Skymapper via TAP

In [29]:
service = vo.dal.TAPService('https://api.skymapper.nci.org.au/public/tap')

In [38]:
# Test print table names

query = """
SELECT table_name
FROM TAP_SCHEMA.tables
"""

tables = service.search(query)

In [ ]:
tablesdf = tables.to_table().to_pandas()
# List of tables
tablesdf.to_csv("tables.txt", index=False, sep="\t", header=0)

# Find start dates for image

In [51]:
from astropy.time import Time

In [58]:
query = """
SELECT DISTINCT night_mjd
FROM dr4.images
ORDER BY night_mjd ASC
"""

results = service.search(query)

In [63]:
datesdf = results.to_table().to_pandas()
datesdf.columns = ["time"]

In [ ]:
datesdf["time"] = Time(datesdf["time"].values, format="mjd") # Turn into astropy time object (modified julian date)
datesdf["date"]  = datesdf["time"].apply(lambda t: t.to_value("iso", subfmt="date"))       # '2023-03-01 00:00:00.000'

In [66]:
print(datesdf)

         time        date
0     56730.0  2014-03-14
1     56731.0  2014-03-15
2     56732.0  2014-03-16
3     56733.0  2014-03-17
4     56734.0  2014-03-18
...       ...         ...
1752  59469.0  2021-09-12
1753  59470.0  2021-09-13
1754  59471.0  2021-09-14
1755  59472.0  2021-09-15
1756  59473.0  2021-09-16

[1757 rows x 2 columns]


## Test: Find the last image, and print out all ccds and their columns to this image

In [130]:
query = """
SELECT TOP 1 image_id
FROM dr4.images
ORDER BY night_mjd DESC
"""
results = service.search(query)
results = results.to_table()

In [132]:
print(results)

   image_id   
--------------
20210916085152


In [133]:
img_id = results.iloc[0]
print(results["image_id"].iloc[0]) # Find image id of last image

ValueError: Can only use TableILoc for a table with indices

In [ ]:
query = f"""
SELECT *
FROM dr4.ccds
WHERE image_id = {img_id}
"""
results = service.search(query)
results = results.to_table()
last_img_ccd = results.to_table().to_pandas()

In [115]:
print(results)
print(results.fieldnames)
for name in results.fieldnames:
    desc = results.getdesc(name)
    print(f"{name:20s}  datatype={desc}  arraysize={desc.arraysize}")
for i in results[0]:
    print(type(i))

<DALResultsTable length=32>
   image_id    ...
               ...
    int64      ...
-------------- ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
           ... ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
20210916085152 ...
('image_id', 'ccd', 'image_type', 'filename', 'maskname', 'image', 'filter', 'mjd_obs', 'fwhm_ccd', 'elong_ccd', 'nsatpix', 'sb_mag', 'phot_nstar', 'header', 'coverage')
image_id              datatype=<FIELD ID="image_id" datatype="long" name="image_id" ucd="meta.id.part"/>  arraysize=None
ccd                   datatype=<FIELD ID="ccd" datatype="int" name="ccd" ucd="meta.id.part"/>  arraysize=None
image_type            datatype=<FIELD ID="image_type" arraysize="256" datatype="char" name="image_type" ucd="meta.code.class"/>  array

In [97]:
last_img_ccd["coverage"].iloc[0]

'POLYGON ICRS 234.28437999999983 -48.49452000000003 233.85915999999992'

# Tests

In [ ]:
query = f"""
SELECT TOP 1 *
FROM dr4.photometry
WHERE image_id = 20210916085152
"""
results = service.search(query)
results = results.to_table()


object_id  object_id_local    image_id    ccd ... img_qual x_mosaic y_mosaic
                                              ...            pix      pix   
---------- --------------- -------------- --- ... -------- -------- --------
2469629440               1 20210916085152   1 ...        3  6803.89 -4111.86


In [136]:
print(results.dtype)

[('object_id', '<i8'), ('object_id_local', '<i8'), ('image_id', '<i8'), ('ccd', '<i2'), ('filter', '<U256'), ('ra_img', '<f8'), ('decl_img', '<f8'), ('x_img', '<f4'), ('y_img', '<f4'), ('flags', '<i2'), ('nimaflags', '<i4'), ('imaflags', '<i4'), ('background', '<f4'), ('flux_max', '<f4'), ('mu_max', '<f4'), ('class_star', '<f4'), ('a', '<f4'), ('e_a', '<f4'), ('b', '<f4'), ('e_b', '<f4'), ('pa', '<f4'), ('e_pa', '<f4'), ('elong', '<f4'), ('fwhm', '<f4'), ('radius_petro', '<f4'), ('radius_kron', '<f4'), ('radius_frac20', '<f4'), ('radius_frac50', '<f4'), ('radius_frac90', '<f4'), ('chi2_psf', '<f4'), ('flux_psf', '<f4'), ('e_flux_psf', '<f4'), ('mag_psf', '<f4'), ('e_mag_psf', '<f4'), ('flux_petro', '<f4'), ('e_flux_petro', '<f4'), ('mag_petro', '<f4'), ('e_mag_petro', '<f4'), ('flux_kron', '<f4'), ('e_flux_kron', '<f4'), ('mag_kron', '<f4'), ('e_mag_kron', '<f4'), ('flux_ap02', '<f4'), ('e_flux_ap02', '<f4'), ('mag_apc02', '<f4'), ('e_mag_apc02', '<f4'), ('flux_ap03', '<f4'), ('e_flux_